# Tutorial: análisis de GW150914

Recorre el pipeline de `gw-analysis` con datos reales de GWOSC. Instala antes el paquete (`pip install -e .`) y ten conexión la primera vez.

> Las métricas son **significancia de exceso de potencia** y un **filtro adaptado educativo** (plantillas solo de inspiral); no reproducen los SNR publicados. Ver `docs/METODOLOGIA.md` y `docs/AULA.md`.

In [ ]:
import matplotlib.pyplot as plt

from gw_analysis.io.downloader import (
    crop_strain,
    download_event_strain,
    event_gps,
    load_hdf5_strain,
)
from gw_analysis.processing.matched_filter import matched_filter_search
from gw_analysis.processing.snr_calculator import analyze_signal, whiten_bandpass
from gw_analysis.viz.plots import plot_full_analysis, plot_snr_timeseries

## 1. Datos
Descargamos H1 y recortamos 64 s alrededor del evento, para que el resultado no dependa de la longitud del archivo.

In [ ]:
event_id, detector = "GW150914", "H1"
gps = event_gps(event_id)
s = crop_strain(load_hdf5_strain(download_event_strain(event_id, detector)), gps, 64)
print(f"{len(s.strain)} muestras a {s.sample_rate:.0f} Hz; evento en GPS {gps}")

## 2. El strain crudo no deja ver nada (Práctica 1)
¿Por qué? Compara con la señal blanqueada y filtrada.

In [ ]:
w = whiten_bandpass(s.strain, s.sample_rate)
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(s.times - gps, s.strain, lw=0.5); ax[0].set_ylabel("strain crudo")
ax[1].plot(s.times - gps, w, lw=0.5); ax[1].set_ylabel("blanqueado + 35-350 Hz")
ax[1].set_xlim(-0.5, 0.3); ax[1].set_xlabel("t - t_evento (s)")
plt.show()

## 3. Exceso de potencia (Práctica 3)
Prueba otros valores de `span` y comprueba cuánto cambian los resultados.

In [ ]:
metrics = analyze_signal(s.times, s.strain, event_gps=gps)
metrics

In [ ]:
png = plot_full_analysis(s.times, s.strain, metrics, event_id)
plt.imshow(plt.imread(png)); plt.axis("off"); plt.show()

## 4. Filtro adaptado y falsa alarma (Práctica 4)

In [ ]:
mf = matched_filter_search(s.times, s.strain, gps)
mf.summary()

In [ ]:
png = plot_snr_timeseries(mf, event_id)
plt.imshow(plt.imread(png)); plt.axis("off"); plt.show()

## Para pensar
1. ¿Por qué el SNR obtenido es menor que el publicado? 
2. ¿Qué significa `false_alarm_probability_lower_bound`?
3. ¿Qué cambiaría con un banco que incluya fusión y ringdown?